### Solving an inverse Problem with Qewton and PINNs
Assume you are jumping from a plane with a parachute. Under some simplifications, your fall can be described by the following ODE:
\begin{align*}
    \begin{split}
        \partial_t^2 u(t) &= {\color{red}{D(t)}} \Big[\partial_t u(t)\Big]^2 - g  \quad 
        \text{ for } t\in (0,T), \\
        u(0) &= H, \\
        \partial_t u(0) &= v_0,
    \end{split}
\end{align*}
where $u : [0, T) \to \mathbb{R}$ is the height at time $t$, $H$ the starting height, $g$ the gravity constant, $v_0$ the initial velocity, and $\color{red}{D(t)}$ a friction coefficient at time $t$.

Given a noisy dataset $\{(u_i, t_i)\}_{i=1}^N$ of the height $u_i$ at time $t_i$, we aim to reconstruct the movement of the parachute described by $\color{red}{D(t)}$.

In [ ]:
# This block is for GPU selection. Please execute.
# import pathlib
# import os
# user = int(str(pathlib.Path().resolve())[22:24])
# os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

path = "/localdata/tomfre/SolutionData/ParachuteData/"

In [ ]:
import qewton

# Here all parameters are defined:
g = 9.81                # Gravity constant
H = 0                   # Initial height H
v_0 = -2                # Initial velocity v_0
t_min, t_max = 0.0, 10 # Bounds for time interval

We start with the basic setup, defining the variables, domains and samplers for the problem.

In [ ]:
T = qewton.Variable('t', dim=1)
U = qewton.Variable('u', dim=1)
# TODO: Create the Variable of the parameter function D
D = qewton.Variable()

# TODO: Define the time interval I_t
I_t = qewton.geometries.Interval()

# TODO: Create a grid sampler for the ODE condition on I_t.
# Use 5000 as the number of sampling points
ode_sampler = qewton.

Now we create two neural networks, one for the solution function $u$ and one for the parameter function $D$.

In [ ]:
# First the model for u, as before:
model_u = qewton.FCN(
    in_neurons=T,
    hidden_neurons=16,
    out_neurons=U,
    n_hidden_layers=2
    )

# TODO: Create a FCN for the parameter function D(t). 
# Use hidden_neurons=16 and n_hidden_layers=2
model_D = qewton.FCN()

Since we consider a parameter identification problem, we need some data to identify the parameter function $D(t)$.
We assume that we have measurements of the solution function $u(t)$ at certain time points.

In [ ]:
# Load the data
t_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "time_data.pt")
u_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "noisy_measurement.pt")

In [ ]:
# Create the data loader for the measurement data
dataset =  qewton.GridDataSet(data=[t_data, u_data], feature_variables=[T, U])

data_loader = qewton.data.DataLoader(
    data_set=dataset,
    batch_size=len(u_data),
    shuffle_data=False,
)

We can plot the given measurement data to visualize it.

In [7]:
plot_config = qewton.config.DataConfiguration(
    qewton.config.GeometryAxes(qewton.geometries.GridGeometry(T, t_data)), qewton.config.FeatureAxes(U)
)
data_plot = qewton.viz.LinePlot(u_data, plot_config, x=T, y=U)
qewton.viz.Figure(data_plot, title="Measurement data of u").show()

In [ ]:
# Now we build the computation graph for the data constraint. 
# The data constraint is a mean squared error (MSE) constraint that compares 
# the model's predictions with the actual measurements:
data_constraint = qewton.constraints.MSEConstraint(weight=10.0)

# Empty pipeline for the data constraint
data_pipeline = qewton.Graph()

with data_pipeline.tracker(0):
    t, u = data_loader()
    model_prediction = # TODO: Evaluate the model for u at time t
    data_constraint(model_prediction, u)

In [ ]:
# TODO: Create a residual for the ODE condition
def ode_residual(t : T, u: U, D: D):
    pass

# We now use a slow ramp for the activation of the ODE constraint
# For the first 2000 iterations will not be taken into account and then later gradually activated
ramp = qewton.CosineRampWeight(2000, 4000)

ode_constraint = qewton.PINNConstraint(
  ode_residual, 
  activation_condition=ramp,
  name="ODEConstraint")

ode_pipeline = qewton.PINNPipeline(
    sampler=ode_sampler,
    constraint=ode_constraint,
    models=[model_u, model_D]
)

After we created the data and physics constraints, we can now combine them into a single training loop to optimize our models.

In [ ]:
# Again a combination of Adam and LBFGS optimization phases for training
# TODO: Set the learning rates and iteration number for both optimizers
# For Adam set: lr=0.001, max_iterations=10000
# For LBFGS set: lr=1.0, max_iterations=500
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam()
)

lbfgs_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.LBFGS(),
    optimizer_args={"max_eval": 10}
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase, lbfgs_phase],
    graphs=[ode_pipeline, data_pipeline],
    training_objectives=[ode_constraint, data_constraint],
    device=qewton.cuda(0)
)

trainer.run()

Finally, we can check how good the reconstructed $D$ is by comparing it to the true value.

In [ ]:
# Now we can check how good the D can be reconstructed. 
# We can plot the learned D(t) and compare it with the true 
# D(t) = 3 * (1.1 + sin(-0.42*t))
import numpy as np

def true_D(t):
    return 3 * (1.1 + np.sin(-0.42*t))

layout = ode_pipeline.visualize(model_D.output_ports[0], reference=true_D)
qewton.viz.Figure(layout).show()